In [1]:
import numpy as np

import csv
import time
import requests
from bs4 import BeautifulSoup




# temp

In [5]:
BASE_URL = "https://www.goodreads.com/book/show/32758901-all-systems-red?page={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}



In [12]:
! pip3 install playwright
! playwright install chromium


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.4/39.4 MB 44.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.1/271.1 kB 24.4 MB/s eta 0:00:00


In [14]:
# import asyncio
# import random
# import pandas as pd
# from bs4 import BeautifulSoup
# from playwright.async_api import async_playwright
# import time

# OUTPUT_CSV = "goodreads_all_systems_red_reviews.csv"


# def human_delay(a=0.3, b=1.4):
#     """Return a random human-like delay."""
#     return random.uniform(a, b)


# async def safe_click(page, selector):
#     """Try clicking a selector if it exists."""
#     try:
#         btn = await page.query_selector(selector)
#         if btn:
#             await btn.click()
#             await page.wait_for_timeout(int(human_delay(500, 1500)))  # 0.5–1.5s delay
#             return True
#     except:
#         pass
#     return False


# async def expand_all_reviews(page):
#     """
#     Continually click 'Show More Reviews' until no more appear.
#     """
#     click_count = 0

#     while True:
#         clicked = await safe_click(page, "button.Button--showMoreReviews")

#         if not clicked:
#             print("✓ No more 'Show More Reviews' buttons. Pagination finished.")
#             break

#         click_count += 1
#         print(f"Clicked 'Show More Reviews' {click_count} times.")

#         # Safety stop
#         if click_count > 5000:
#             print("⚠ Safety stop — too many clicks.")
#             break


# async def expand_long_review_texts(page):
#     """
#     Clicks all 'More' buttons inside individual reviews
#     so that full review text is visible.
#     """
#     more_buttons = await page.query_selector_all("span.ReadMore__link")

#     print(f"Found {len(more_buttons)} 'More' buttons inside reviews.")

#     for i, btn in enumerate(more_buttons):
#         try:
#             await btn.scroll_into_view_if_needed()
#             await page.wait_for_timeout(int(human_delay(200, 500)))
#             await btn.click()
#             await page.wait_for_timeout(int(human_delay(400, 1200)))
#             print(f"Expanded review full text #{i+1}")
#         except:
#             print(f"Failed to expand review #{i+1}")


# def parse_reviews(html):
#     soup = BeautifulSoup(html, "html.parser")
#     results = []

#     review_blocks = soup.select("div.review")

#     for r in review_blocks:
#         # Username
#         user = r.select_one("a.user")
#         username = user.get_text(strip=True) if user else ""

#         # Date
#         date_tag = r.select_one("a.reviewDate")
#         date = date_tag.get_text(strip=True) if date_tag else ""

#         # Rating
#         rating_tag = r.select_one("span.staticStars:not(.notranslate)")
#         rating = rating_tag["title"] if rating_tag and rating_tag.has_attr("title") else ""

#         # Full review text
#         # After expanding, the deepest <span> has the complete text.
#         text_tag = r.select_one("div.reviewText span.readable span:nth-of-type(2)")
#         if not text_tag:
#             text_tag = r.select_one("div.reviewText span.readable span")

#         review_text = text_tag.get_text(strip=True, separator=" ") if text_tag else ""

#         results.append({
#             "username": username,
#             "date": date,
#             "rating": rating,
#             "review_text": review_text
#         })

#     return results


# async def scrape_goodreads():
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True)
#         page = await browser.new_page()

#         url = "https://www.goodreads.com/book/show/32758901-all-systems-red"
#         print("Loading:", url)
#         await page.goto(url, timeout=60000)
#         await page.wait_for_timeout(int(human_delay(800, 1500)))

#         # Add small random scrolls (reduces bot detection)
#         for _ in range(3):
#             await page.mouse.wheel(0, random.randint(200, 800))
#             await page.wait_for_timeout(int(human_delay(300, 900)))

#         # Step 1 — keep hitting "Show More Reviews"
#         await expand_all_reviews(page)

#         # Step 2 — expand all long review texts
#         await expand_long_review_texts(page)

#         # Step 3 — extract everything
#         html = await page.content()
#         reviews = parse_reviews(html)

#         df = pd.DataFrame(reviews)
#         df.to_csv(OUTPUT_CSV, index=False)

#         print(f"\n🎉 DONE! Extracted {len(df)} reviews.")
#         print(f"Saved to: {OUTPUT_CSV}")

#         await browser.close()


# await scrape_goodreads_test()


NameError: name 'scrape_goodreads_test' is not defined

In [16]:
import asyncio
import random
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

OUTPUT_CSV = "goodreads_test_fixed.csv"


def human_delay(a=0.2, b=1.0):
    return random.uniform(a, b)


async def expand_pages_limit(page, max_pages=5):
    clicks = 0

    while clicks < max_pages:
        btn = await page.query_selector("button[data-testid='reviews-show-more']")
        if not btn:
            print("No more 'Show More Reviews' button.")
            break

        await btn.click()
        print(f"Loaded page {clicks+1}")
        clicks += 1

        await page.wait_for_timeout(int(human_delay(800, 1500)))

    print("Finished expanding pages.")


async def expand_individual_reviews(page):
    buttons = await page.query_selector_all("button[data-testid='review-show-more']")
    print(f"Found {len(buttons)} long-review expand buttons")

    for i, btn in enumerate(buttons):
        try:
            await btn.scroll_into_view_if_needed()
            await page.wait_for_timeout(int(human_delay(200, 400)))
            await btn.click()
            await page.wait_for_timeout(int(human_delay(400, 800)))
        except:
            pass


def parse_reviews(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []

    blocks = soup.select("div[data-testid='review']")

    for r in blocks:
        # Username
        user = r.select_one("a[data-testid='name']")
        username = user.get_text(strip=True) if user else ""

        # Date
        date = r.select_one("span[data-testid='review-date']")
        date = date.get_text(strip=True) if date else ""

        # Rating
        stars = r.select_one("span[data-testid='stars']")
        rating = stars["title"] if stars and stars.has_attr("title") else ""

        # Review text
        body = r.select_one("div[data-testid='review-body']")
        text = body.get_text(" ", strip=True) if body else ""

        results.append({
            "username": username,
            "date": date,
            "rating": rating,
            "review_text": text,
        })

    return results


# async def scrape_goodreads_test():
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True)
#         page = await browser.new_page()

#         url = "https://www.goodreads.com/book/show/32758901-all-systems-red"
#         print("Loading:", url)
#         await page.goto(url, timeout=60000)

#         # Load 5 pages worth of reviews
#         await expand_pages_limit(page, max_pages=5)

#         # Expand long reviews
#         await expand_individual_reviews(page)

#         # Parse
#         html = await page.content()
#         reviews = parse_reviews(html)

#         df = pd.DataFrame(reviews)
#         df.to_csv(OUTPUT_CSV, index=False)

#         print(f"\nExtracted {len(df)} reviews.")
#         print("Saved:", OUTPUT_CSV)

#         await browser.close()


# # FOR JUPYTER (NO asyncio.run)
# await scrape_goodreads_test()


Loading: https://www.goodreads.com/book/show/32758901-all-systems-red
No more 'Show More Reviews' button.
Finished expanding pages.
Found 0 long-review expand buttons

Extracted 0 reviews.
Saved: goodreads_test_fixed.csv


In [22]:
import json


JSONDecodeError: Extra data: line 2 column 1 (char 1345)

In [13]:
# data from UCSD 
# https://sites.google.com/eng.ucsd.edu/ucsdbookgraph/home

MB_book_info = None
MB_book_id = None

with open("./data/goodreads_books.json", "r") as f:
    for line in f:
        elem = json.loads(line)
        if 'All Systems Red' in elem.get('title', ''):
            MB_book_info = elem
            MB_book_id = elem['book_id']
            break


{'isbn': '0765397536', 'text_reviews_count': '67', 'series': ['980193'], 'country_code': 'US', 'language_code': 'eng', 'popular_shelves': [{'count': '8745', 'name': 'to-read'}, {'count': '491', 'name': 'currently-reading'}, {'count': '346', 'name': 'science-fiction'}, {'count': '285', 'name': 'sci-fi'}, {'count': '96', 'name': 'fiction'}, {'count': '82', 'name': 'novella'}, {'count': '66', 'name': 'scifi'}, {'count': '47', 'name': 'sf'}, {'count': '46', 'name': 'read-in-2017'}, {'count': '42', 'name': 'favorites'}, {'count': '39', 'name': 'ebook'}, {'count': '33', 'name': 'series'}, {'count': '25', 'name': 'short-stories'}, {'count': '24', 'name': 'kindle'}, {'count': '23', 'name': 'sff'}, {'count': '23', 'name': 'adult'}, {'count': '22', 'name': 'read-2017'}, {'count': '18', 'name': 'novellas'}, {'count': '18', 'name': 'robots'}, {'count': '18', 'name': 'speculative-fiction'}, {'count': '17', 'name': 'space'}, {'count': '17', 'name': 'sci-fi-fantasy'}, {'count': '16', 'name': 'artific

In [14]:

print(MB_book_info)
print(MB_book_id)

{'isbn': '0765397536', 'text_reviews_count': '67', 'series': ['980193'], 'country_code': 'US', 'language_code': 'eng', 'popular_shelves': [{'count': '8745', 'name': 'to-read'}, {'count': '491', 'name': 'currently-reading'}, {'count': '346', 'name': 'science-fiction'}, {'count': '285', 'name': 'sci-fi'}, {'count': '96', 'name': 'fiction'}, {'count': '82', 'name': 'novella'}, {'count': '66', 'name': 'scifi'}, {'count': '47', 'name': 'sf'}, {'count': '46', 'name': 'read-in-2017'}, {'count': '42', 'name': 'favorites'}, {'count': '39', 'name': 'ebook'}, {'count': '33', 'name': 'series'}, {'count': '25', 'name': 'short-stories'}, {'count': '24', 'name': 'kindle'}, {'count': '23', 'name': 'sff'}, {'count': '23', 'name': 'adult'}, {'count': '22', 'name': 'read-2017'}, {'count': '18', 'name': 'novellas'}, {'count': '18', 'name': 'robots'}, {'count': '18', 'name': 'speculative-fiction'}, {'count': '17', 'name': 'space'}, {'count': '17', 'name': 'sci-fi-fantasy'}, {'count': '16', 'name': 'artific

In [22]:
MB_reviews = []

with open("./data/goodreads_reviews_dedup.json", "r") as f:
    for line in f:
        item = json.loads(line)
        if item.get("book_id") == MB_book_id:
            MB_reviews.append(item)


In [24]:
len(MB_reviews)

21

In [25]:
MB_reviews

[{'user_id': 'edc68f0f9e163d47b1503b3cdc4e2c5e',
  'book_id': '33387769',
  'review_id': '8d235f28b8a158b50d800eceb8e62647',
  'rating': 5,
  'review_text': 'I loved this SO MUCH! \n Further thoughts: http://ladybusiness.dreamwidth.org/20...',
  'date_added': 'Wed Jan 04 19:02:26 -0800 2017',
  'date_updated': 'Mon May 15 20:37:18 -0700 2017',
  'read_at': 'Tue May 09 00:00:00 -0700 2017',
  'started_at': '',
  'n_votes': 7,
  'n_comments': 0},
 {'user_id': '2d83dd909c236f532cd423e26f85bcc9',
  'book_id': '33387769',
  'review_id': 'f573e5e49f711913c65f052a86e0f0a1',
  'rating': 5,
  'review_text': 'I really loved this. Fun space opera drama with a snarky, anti-social protag, plus a surprisingly rich underlying discussion of what makes a person a person. I really hope we see all these characters many times over again.',
  'date_added': 'Wed Oct 26 05:31:46 -0700 2016',
  'date_updated': 'Sat Aug 19 11:39:09 -0700 2017',
  'read_at': 'Fri Aug 18 00:00:00 -0700 2017',
  'started_at': 'Fr

In [ ]:
new_MB_reviews = []

for elem in reviews:
    if elem['book_id'] == MB_book_id:
        new_MB_reviews.append(elem)

In [19]:
len(MB_reviews[0])

29

In [21]:
print(MB_reviews[0])

{'isbn': '0765397536', 'text_reviews_count': '67', 'series': ['980193'], 'country_code': 'US', 'language_code': 'eng', 'popular_shelves': [{'count': '8745', 'name': 'to-read'}, {'count': '491', 'name': 'currently-reading'}, {'count': '346', 'name': 'science-fiction'}, {'count': '285', 'name': 'sci-fi'}, {'count': '96', 'name': 'fiction'}, {'count': '82', 'name': 'novella'}, {'count': '66', 'name': 'scifi'}, {'count': '47', 'name': 'sf'}, {'count': '46', 'name': 'read-in-2017'}, {'count': '42', 'name': 'favorites'}, {'count': '39', 'name': 'ebook'}, {'count': '33', 'name': 'series'}, {'count': '25', 'name': 'short-stories'}, {'count': '24', 'name': 'kindle'}, {'count': '23', 'name': 'sff'}, {'count': '23', 'name': 'adult'}, {'count': '22', 'name': 'read-2017'}, {'count': '18', 'name': 'novellas'}, {'count': '18', 'name': 'robots'}, {'count': '18', 'name': 'speculative-fiction'}, {'count': '17', 'name': 'space'}, {'count': '17', 'name': 'sci-fi-fantasy'}, {'count': '16', 'name': 'artific

In [ ]:
# books = []

# with open("./data/goodreads_books.json", "r") as f:
#     for i, line in enumerate(f):
#         if i == 5:    # only first 5 rows
#             break
#         books.append(json.loads(line))

# print(books)